# D-V6 — Chạy lại Stage 2 Gán nhãn SGOD bằng vLLM nội bộ trên Colab T4

Notebook này chạy lại `datagen/label_scenarios.py` (Stage 2) phục vụ sửa lỗi mất tham số cho bộ dữ liệu SGOD `train.jsonl` (Task 10 D-V6).

### 💡 Đặc điểm nổi bật:
- **100% Miễn phí, KHÔNG cần API key OpenAI / Mentor:** Tự khởi động máy chủ `vLLM` phục vụ mô hình `Qwen/Qwen2.5-3B-Instruct` trực tiếp trên GPU T4 của Colab.
- **Không tốn token ngoài, không bị rate-limit (429).**
- **Chuẩn bị duy nhất:** Thêm `GITHUB_TOKEN` vào mục **Secrets (biểu tượng 🔑)** trên Colab (quyền `repo`), bật **Notebook access**.
- **Runtime:** Hãy chắc chắn bạn đã chọn `Runtime -> Change runtime type -> T4 GPU`.

In [ ]:
# --- Ô 1: Thiết lập môi trường (chuẩn _setup_snippet.md) --- #
import os, shlex, subprocess, sys
from pathlib import Path

REPO_HTTPS = "github.com/thanhhao98/ChatSystem"
MARKER = "docs/contracts/cli.md"

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

def _github_token():
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    if IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    return os.environ.get("GITHUB_TOKEN", "")

if IN_COLAB or IN_KAGGLE:
    try:
        _token = _github_token()
    except Exception as e:
        raise RuntimeError("Thiếu secret GITHUB_TOKEN. Colab: biểu tượng chìa khoá -> Secrets -> Add GITHUB_TOKEN") from e
    if not Path("ChatSystem").exists():
        _r = subprocess.run(
            ["git", "clone", "--quiet", f"https://{_token}@{REPO_HTTPS}",
             "ChatSystem", "--branch", "data/fix-empty-params-thehy"],
            capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError("git clone thất bại: " + _r.stderr.replace(_token, "<token>"))
    os.chdir("ChatSystem")
    del _token
else:
    _here = Path.cwd().resolve()
    for _cand in [_here, *_here.parents]:
        if (_cand / MARKER).exists():
            os.chdir(_cand); break
    else:
        raise FileNotFoundError(f"Không tìm thấy gốc repo từ {_here}")

REPO = Path.cwd()

def _git(*args):
    try:
        return subprocess.run(["git", *args], cwd=REPO, capture_output=True, text=True).stdout.strip()
    except OSError:
        return ""

GIT_SHA = _git("rev-parse", "--short", "HEAD") or "no-git"
AUTHOR = os.environ.get("GITHUB_USER") or _git("config", "user.name") or "Thehy-IT"

def run(cmd):
    shown = " ".join(shlex.quote(c) for c in cmd).replace(shlex.quote(sys.executable), "python", 1)
    print("$ " + shown)
    p = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
    if p.stdout: print(p.stdout.rstrip())
    if p.stderr: print(p.stderr.rstrip())
    print(f"[exit code = {p.returncode}]")
    return p

print(f"REPO      = {REPO}")
print(f"git HEAD  = {GIT_SHA}")
print(f"python    = {sys.version.split()[0]} · Colab = {IN_COLAB} · author = {AUTHOR}")

In [ ]:
# --- Ô 2: Cài đặt vLLM và các thư viện cần thiết --- #
import subprocess, sys

print(">>> Đang cài đặt vllm và openai phiên bản tương thích...")
# Cài đặt vllm và openai mới nhất (vllm yêu cầu openai >= 2.25.0)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts", "vllm", "openai", "requests", "pyyaml"], check=True)
print("✅ Cài đặt hoàn tất! Kiểm tra GPU khả dụng:")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"])

In [ ]:
# --- Ô 3: Khởi động máy chủ vLLM phục vụ Qwen2.5-3B-Instruct trên GPU T4 --- #
import subprocess, time, urllib.request, json, os, sys

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
PORT = 8000
LOG_FILE = "vllm_server.log"

print(">>> Dọn dẹp tiến trình cũ...")
subprocess.run(["fuser", "-k", f"{PORT}/tcp"], capture_output=True)
subprocess.run(["pkill", "-9", "-f", "vllm"], capture_output=True)
time.sleep(2)

env = os.environ.copy()
env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
env["VLLM_USE_V1"] = "0"
env["CUDA_VISIBLE_DEVICES"] = "0"

cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL_NAME,
    "--port", str(PORT),
    "--max-model-len", "2048",
    "--gpu-memory-utilization", "0.85",
    "--dtype", "float16",
    "--enforce-eager",
    "--disable-log-requests"
]

print(f">>> Đang khởi động vLLM Server phục vụ '{MODEL_NAME}' trên port {PORT}...")
log_f = open(LOG_FILE, "w", encoding="utf-8")
proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, env=env)

start_time = time.time()
ready = False
for _ in range(60):
    time.sleep(5)
    if proc.poll() is not None:
        print(f"❌ [FATAL] vLLM Server thoát đột ngột với exit code = {proc.returncode}!")
        break
    try:
        with urllib.request.urlopen(f"http://localhost:{PORT}/v1/models", timeout=3) as resp:
            if resp.status == 200:
                models_info = json.loads(resp.read().decode("utf-8"))
                ready = True
                elapsed = time.time() - start_time
                print(f"✅ [THÀNH CÔNG] vLLM Server đã sẵn sàng phục vụ sau {elapsed:.1f}s!")
                print("   Models khả dụng:", [m["id"] for m in models_info.get("data", [])])
                break
    except Exception:
        print(".", end="", flush=True)

if not ready:
    print(f"\n❌ Không thể kết nối tới vLLM Server. 35 dòng log cuối:")
    with open(LOG_FILE, "r", encoding="utf-8", errors="ignore") as f:
        print("".join(f.readlines()[-35:]))
    raise RuntimeError("vLLM Server khởi động thất bại!")

In [ ]:
# --- Ô 4: Thiết lập biến môi trường nội bộ và kiểm tra bản vá --- #
import os, json
from pathlib import Path

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
os.environ["OPENAI_API_KEY"] = "token-local-vllm"
os.environ["OPENAI_BASE_URL"] = "http://localhost:8000/v1"
os.environ["GEN_MODEL"] = MODEL_NAME
os.environ["LABEL_MODELS"] = MODEL_NAME
os.environ["ARBITER_MODEL"] = MODEL_NAME

content = Path("datagen/label_scenarios.py").read_text(encoding="utf-8")
assert 'd.get("parameters")' in content, "❌ LỖI: Chưa có bản vá trong datagen/label_scenarios.py! Kiểm tra nhánh git."
print("✅ Bản vá 'parameters' fallback đã có hiệu lực.")

raw_path = Path("data/sgod/raw.json")
assert raw_path.exists(), f"❌ Không tìm thấy {raw_path}"
raw_data = json.loads(raw_path.read_text(encoding="utf-8"))
print(f"✅ File raw.json sẵn sàng với {len(raw_data)} bản ghi.")

In [ ]:
# --- Ô 5: Chạy Stage 2 gán nhãn qua vLLM nội bộ --- #
from pathlib import Path

labeled_path = Path("data/sgod/labeled.json")
if labeled_path.exists():
    labeled_path.rename("data/sgod/labeled_v1_buggy.json")
    print("🗂️ Đã sao lưu labeled.json cũ -> labeled_v1_buggy.json")

print("🚀 Bắt đầu gán nhãn dữ liệu (Stage 2) qua vLLM...")
res = run([
    sys.executable, "datagen/label_scenarios.py",
    "--input", "data/sgod/raw.json",
    "--output", "data/sgod/labeled.json",
    "--report", "data/sgod/label_report_v2.json",
    "--models", "Qwen/Qwen2.5-3B-Instruct",
    "--workers", "8"
])
assert res.returncode == 0, f"❌ Stage 2 thất bại! Exit code = {res.returncode}"
print("✅ Stage 2 hoàn tất thành công!")

In [ ]:
# --- Ô 6: Đối chứng kết quả & Tự động lưu lên Google Drive --- #
import json
from pathlib import Path

data = json.loads(Path("data/sgod/labeled.json").read_text(encoding="utf-8"))
has_p = sum(1 for d in data if d.get("expected_params") and d.get("expected_tool"))
empty_p = sum(1 for d in data if d.get("expected_tool") and not d.get("expected_params"))
total = len(data)

print(f"📊 Kết quả kiểm tra data/sgod/labeled.json:")
print(f"   Tổng số dòng: {total}")
print(f"   Dòng có tham số: {has_p} ({has_p/max(total,1)*100:.1f}%)")
print(f"   Dòng rỗng tham số: {empty_p} ({empty_p/max(total,1)*100:.1f}%)")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    import shutil, datetime
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    dst_dir = Path("/content/drive/MyDrive/sgod_D_V6")
    dst_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy("data/sgod/labeled.json", dst_dir / f"{ts}_labeled.json")
    shutil.copy("data/sgod/label_report_v2.json", dst_dir / f"{ts}_label_report_v2.json")
    print(f"✅ Đã lưu bản sao lên Google Drive tại: {dst_dir}/{ts}_labeled.json")

In [ ]:
# --- Ô 7: Tạo khối báo cáo ClickUp chuẩn --- #
import datetime, json
from pathlib import Path

data = json.loads(Path("data/sgod/labeled.json").read_text(encoding="utf-8"))
raw = json.loads(Path("data/sgod/raw.json").read_text(encoding="utf-8"))
has_p = sum(1 for d in data if d.get("expected_params") and d.get("expected_tool"))
empty_p = sum(1 for d in data if d.get("expected_tool") and not d.get("expected_params"))

TASK = "D-V6 Fix tham số rỗng train.jsonl (Stage 2 Relabel)"
lines = [
    f"## Báo cáo {TASK} — {datetime.date.today().isoformat()} — {AUTHOR}",
    f"- Notebook: `notebooks/data/04_relabel_fix_params.ipynb` @ `{GIT_SHA}` · môi trường: {'Colab T4 (vLLM local)' if IN_COLAB else 'local'}",
    f"- **Hạ tầng phục vụ:** `vLLM` server nội bộ chạy model `Qwen/Qwen2.5-3B-Instruct` trên port 8000 (0 token phí ngoài).",
    f"- **Stage 2 kết quả:** labeled.json total={len(data)}, has_params={has_p} ({has_p/max(len(data),1)*100:.1f}%), empty={empty_p}",
    f"- **Fix:** `d.get('parameters') or d.get('params')` tại `datagen/label_scenarios.py`",
    f"- **Đối chứng:** Trước fix has_params=0 (0%) -> Sau fix has_params={has_p} ({has_p/max(len(data),1)*100:.1f}%)",
    f"- raw.json: {len(raw)} rows | labeled.json: {len(data)} rows",
]
print("\n".join(lines))